Generate Data in Dataset

In [ ]:
import numpy as np

N = 10000
D = 2

X_train = np.random.rand(N, D).astype(np.float32)
y_train = np.random.randint(0, 2, N)

X_test = np.random.rand(1, D).astype(np.float32)

print("Dataset ready!")

Dataset ready!


CPU KNN Implementation


In [ ]:
import time

def knn_cpu(X_train, y_train, X_test, k=3):
    distances = []

    for i in range(len(X_train)):
        distance = np.sqrt(np.sum((X_train[i] - X_test[0])**2))
        distances.append((distance, y_train[i]))

    distances.sort(key=lambda x: x[0])
    neighbors = [label for (_, label) in distances[:k]]

    return max(set(neighbors), key=neighbors.count)

start = time.time()

for _ in range(10):
  prediction = knn_cpu(X_train, y_train, X_test)
  end = time.time()
  cpu_time = (end - start) / 10
  print("Average CPU Time:", cpu_time)

Average CPU Time: 0.006579995155334473
Average CPU Time: 0.012606263160705566
Average CPU Time: 0.018578505516052245
Average CPU Time: 0.024529623985290527
Average CPU Time: 0.030945944786071777
Average CPU Time: 0.037009167671203616
Average CPU Time: 0.04308552742004394
Average CPU Time: 0.04908874034881592
Average CPU Time: 0.0566460132598877
Average CPU Time: 0.06280281543731689


In [ ]:
%%writefile knn_cuda.cu
#include <stdio.h>
#include <math.h>
#include <chrono>

__global__ void compute_distances(float *X_train, float *X_test, float *distances, int N, int D) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;

    if (idx < N) {
        float sum = 0.0;

        for (int j = 0; j < D; j++) {
            float diff = X_train[idx * D + j] - X_test[j];
            sum += diff * diff;
        }

        distances[idx] = sqrtf(sum);
    }
}

int main() {
    int N = 10000;
    int D = 2;

    size_t size_train = N * D * sizeof(float);
    size_t size_test = D * sizeof(float);
    size_t size_dist = N * sizeof(float);

    float *h_X_train = (float*)malloc(size_train);
    float *h_X_test = (float*)malloc(size_test);
    float *h_distances = (float*)malloc(size_dist);

    // Initialize data
    for (int i = 0; i < N * D; i++)
        h_X_train[i] = (float)rand() / RAND_MAX;

    for (int i = 0; i < D; i++)
        h_X_test[i] = (float)rand() / RAND_MAX;

    float *d_X_train, *d_X_test, *d_distances;

    cudaMalloc(&d_X_train, size_train);
    cudaMalloc(&d_X_test, size_test);
    cudaMalloc(&d_distances, size_dist);

    cudaMemcpy(d_X_train, h_X_train, size_train, cudaMemcpyHostToDevice);
    cudaMemcpy(d_X_test, h_X_test, size_test, cudaMemcpyHostToDevice);

    int blockSize = 256;
    int numBlocks = (N + blockSize - 1) / blockSize;


   auto start = std::chrono::high_resolution_clock::now();

   compute_distances<<<numBlocks, blockSize>>>(d_X_train, d_X_test, d_distances, N, D);

   cudaDeviceSynchronize();  // VERY IMPORTANT


   auto end = std::chrono::high_resolution_clock::now();

   std::chrono::duration<double> elapsed = end - start;

   printf("GPU Time: %f seconds\n", elapsed.count());


   cudaMemcpy(h_distances, d_distances, size_dist, cudaMemcpyDeviceToHost);

    printf("First 5 distances:\n");
    for (int i = 0; i < 5; i++)
        printf("%f\n", h_distances[i]);

    cudaFree(d_X_train);
    cudaFree(d_X_test);
    cudaFree(d_distances);

    free(h_X_train);
    free(h_X_test);
    free(h_distances);

    return 0;
}

Overwriting knn_cuda.cu


In [ ]:
!nvcc knn_cuda.cu -o knn_cuda

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [ ]:
!./knn_cuda

GPU Time: 0.000200 seconds
First 5 distances:
0.663863
0.396972
0.861596
0.134567
0.354552
